In [ ]:
import scanpy as sc 
import pandas as pd 
import numpy as np 

import matplotlib.pyplot as plt 
import seaborn as sns 

from sklearn.metrics import adjusted_rand_score
from sklearn.decomposition import PCA

import scipy.sparse as sp 
import warnings
import os

warnings.filterwarnings("ignore")

import sys
sys.path.append('/home/waas/spCLUE-main')

import spCLUE
from spCLUE.simple_ae import train_spatial_ae 

spCLUE.fix_seed(0)

sample_name = "151672"
n_clusters = 5 if sample_name in [str(151669 + x) for x in range(4)] else 7

base_dir = "/home/waas/data/DLPFC"
adata = sc.read_visium(f"{base_dir}/{sample_name}")

label_df = pd.read_csv(f"{base_dir}/{sample_name}_truth.txt", index_col=0, header=None, sep="\t")
label_df = label_df.reindex(adata.obs_names)
adata.obs["Region"] = label_df.iloc[:,0].values

adata.var_names_make_unique()
print(adata)

adata = spCLUE.preprocess(adata)
adata.obsm["X_pca"] = PCA(n_components=200, random_state=0).fit_transform(adata.X)

g_spatial = spCLUE.prepare_graph(adata, "spatial")
adata = train_spatial_ae(adata, g_spatial)

pca = adata.obsm["X_pca"]
ae = adata.obsm["AE"]

var_pca = np.var(pca, axis=0)
var_ae = np.var(ae, axis=0)

def heuristic_gate(pca_var, ae_var):
    g = pca_var / (pca_var + ae_var + 1e-6)
    g = np.clip(g, 0.2, 0.8)
    return g

g = heuristic_gate(var_pca, var_ae)
adata.obsm["AE_final"] = g * pca + (1 - g) * ae

g_expr = spCLUE.prepare_graph(adata, "expr")
graph_dict = {"spatial": g_spatial, "expr": g_expr}


spCLUE_model = spCLUE.spCLUE(
    adata.obsm["AE_final"], 
    graph_dict, 
    n_clusters
)
_, adata.obsm["spCLUE"] = spCLUE_model.train()

refinement = True
cluster_method = "mclust"
pred = spCLUE.clustering(
    adata,
    n_clusters,
    key="spCLUE",
    refinement=refinement,
    cluster_methods=cluster_method,
)

adata = adata[adata.obs.Region.notna()]
ARI = adjusted_rand_score(adata.obs["Region"], adata.obs["mclust_refined"])
print(f"ARI on {sample_name}: {ARI}")

adata.obs["spCLUE"] = adata.obs["mclust_refined"]
sc.pl.spatial(adata, color=["Region", "spCLUE"], title=["Manual Annotation", f"Ours (ARI={ARI:.2f})"])